In [1]:
import sys
!{sys.executable} -m pip install --upgrade jupyter_dash dash dash-leaflet plotly pandas matplotlib pymongo
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import pymongo

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

#username = "aacuser"
#password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(role='admin')

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

df_csv = pd.read_csv("aac_shelter_outcomes.csv")

for record in df_csv.to_dict(orient='records'):
    db.create(record)

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'ProjectTwoLogo.png' # Logo file
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div(
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), height=200, width=200), style={'text-align' : 'center'}
    ),
    html.Center(html.B(html.H1('Austin\'s CS-340 Dashboard'))),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            id = 'filter-type',
            options = [
                {'label':'Water Rescue', 'value' : 'water'},
                {'label':'Mountain or Wilderness Rescue', 'value' : 'mountain'},
                {'label':'Disaster or Individual Tracking', 'value' : 'disaster'},
                {'label':'Reset', 'value' : 'reset'},
                {'label':'Young Dogs', 'value':'young'},
                {'label':'Retriever Breeds', 'value':'retriever'}
            ]
        )
        

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         row_selectable="single",
                         filter_action="native",
                         sort_action="native",
                         page_size= 10,

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
            Output('datatable-id','data'),
            Output('datatable-id','columns'),
            [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type =='water':
        df = pd.DataFrame.from_records(db.read({
            'animal_type' : 'Dog',
            'breed' : {'$in': ['Labrador Retriever Mix', 'Chesa Bay Retr', 'Newfoundland']},
            'sex_upon_outcome' : 'Intact Female',
            'age_upon_outcome_in_weeks' : {'$gte' : 26.0, '$lte' : 156.0}
        }))
    elif filter_type == 'mountain':
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepard","Alaskan Malamute","Old English Sheepdog", 
                              "Siberian Husky", "Rottweiler"
                             ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte":26.0, "$lte":156.0}
            }))
    elif filter_type == 'disaster':
        df = pd.DataFrame.from_records(db.read({
                "animal_type": "Dog",
                "breed": {"$in": ["Doberman Pinscher","German Shepard","Golden Retriever", 
                                  "Bloodhound","Rottweiler"
                                 ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte":20.0, "$lte":300.0}
            }))
    elif filter_type == 'young':
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "age_upon_outcome_in_weeks": {"$lt": 52}
    }))
    elif filter_type == 'retriever':
        df = pd.DataFrame.from_records(db.read({
            "breed": {"$regex": "Retriever", "$options": "i"}
    }))
# Reset filter
    else:
         df = pd.DataFrame.from_records(db.read({}))
        
    if '_id' in df.columns:
        df.drop(columns=['_id'],inplace=True)
    
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data=df.to_dict('records')
        
        
    return (data,columns)


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    if not viewData:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    
    figure = px.pie(dff, names='breed', hole=.3)
    figure.update_layout(legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.99))
    
    return [
        dcc.Graph(
            figure=figure,
            style={'width':'700px', 'height':'500px'}
        )
    ]
    
        
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(#test3
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
    Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return []
    elif index is None:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index)==0:
        row = 0
    else: 
        row = index[0]
        
    popupName = dff['name'].iloc[row]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px', 'margin-left': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(popupName),
                    #html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run(debug=True, port=8050)
